In [10]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import pickle
import joblib
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import PassiveAggressiveClassifier

print("All libraries imported")

All libraries imported


In [11]:
# Cell 2: Load Data and Vectorizer
print("Loading data and vectorizer...")

df = pd.read_csv('../data/processed_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")

with open('tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
print("Vectorizer loaded")

X = vectorizer.transform(df['text'])
y = df['label']
print(f"Vectorized data shape: {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print("Data preparation complete")

Loading data and vectorizer...
Dataset shape: (8501, 8)
Label distribution:
label
1    7202
0    1299
Name: count, dtype: int64
Vectorizer loaded
Vectorized data shape: (8501, 5000)
Training set: (6800, 5000)
Test set: (1701, 5000)
Data preparation complete


In [12]:
# Cell 3: Naive Bayes
print("Training Naive Bayes...")
start_time = time.time()

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
nb_accuracy = accuracy_score(y_test, nb_pred)
nb_time = time.time() - start_time

print(f"Naive Bayes Accuracy: {nb_accuracy:.4f} ({nb_accuracy*100:.2f}%)")
print(f"Training time: {nb_time:.2f} seconds")
joblib.dump(nb_model, 'naive_bayes.pkl')
print("Model saved")

Training Naive Bayes...
Naive Bayes Accuracy: 0.9359 (93.59%)
Training time: 0.03 seconds
Model saved


In [13]:
# Cell 4: Random Forest
print("Training Random Forest...")
start_time = time.time()

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_time = time.time() - start_time

print(f"Random Forest Accuracy: {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)")
print(f"Training time: {rf_time:.2f} seconds")
joblib.dump(rf_model, 'random_forest.pkl')
print("Model saved")

Training Random Forest...
Random Forest Accuracy: 0.9394 (93.94%)
Training time: 1.48 seconds
Model saved


In [14]:
# Cell 5: LightGBM
print("Training LightGBM...")
start_time = time.time()

lgb_model = LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbose=0)
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)
lgb_accuracy = accuracy_score(y_test, lgb_pred)
lgb_time = time.time() - start_time

print(f"LightGBM Accuracy: {lgb_accuracy:.4f} ({lgb_accuracy*100:.2f}%)")
print(f"Training time: {lgb_time:.2f} seconds")
joblib.dump(lgb_model, 'lightgbm.pkl')
print("Model saved")

Training LightGBM...
LightGBM Accuracy: 0.9541 (95.41%)
Training time: 6.06 seconds
Model saved


C:\Users\imtia\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [15]:
# Cell 6: Complement Naive Bayes
print("Training Complement Naive Bayes...")
start_time = time.time()

cnb_model = ComplementNB()
cnb_model.fit(X_train, y_train)
cnb_pred = cnb_model.predict(X_test)
cnb_accuracy = accuracy_score(y_test, cnb_pred)
cnb_time = time.time() - start_time

print(f"ComplementNB Accuracy: {cnb_accuracy:.4f} ({cnb_accuracy*100:.2f}%)")
print(f"Training time: {cnb_time:.2f} seconds")
joblib.dump(cnb_model, 'complement_nb.pkl')
print("Model saved")

Training Complement Naive Bayes...
ComplementNB Accuracy: 0.8895 (88.95%)
Training time: 0.01 seconds
Model saved


In [16]:
# Cell 7: Passive Aggressive
print("Training Passive Aggressive...")
start_time = time.time()

pa_model = PassiveAggressiveClassifier(max_iter=1000, random_state=42, verbose=0)
pa_model.fit(X_train, y_train)
pa_pred = pa_model.predict(X_test)
pa_accuracy = accuracy_score(y_test, pa_pred)
pa_time = time.time() - start_time

print(f"Passive Aggressive Accuracy: {pa_accuracy:.4f} ({pa_accuracy*100:.2f}%)")
print(f"Training time: {pa_time:.2f} seconds")
joblib.dump(pa_model, 'passive_aggressive.pkl')
print("Model saved")

Training Passive Aggressive...
Passive Aggressive Accuracy: 0.9447 (94.47%)
Training time: 0.06 seconds
Model saved


In [17]:
# Cell 8: Results Comparison
print("Model Comparison Results:")
print("=" * 50)

results = {
    'Naive_Bayes': (nb_accuracy, nb_time),
    'Random_Forest': (rf_accuracy, rf_time),
    'LightGBM': (lgb_accuracy, lgb_time),
    'Complement_NB': (cnb_accuracy, cnb_time),
    'Passive_Aggressive': (pa_accuracy, pa_time)
}

print(f"{'Model':<20} {'Accuracy':<12} {'Time (s)':<10}")
print("-" * 50)

for model_name, (accuracy, train_time) in results.items():
    print(f"{model_name:<20} {accuracy:.4f} ({accuracy*100:.2f}%)  {train_time:<8.2f}")

best_model = max(results.items(), key=lambda x: x[1][0])
print(f"\nBest Model: {best_model[0]} ({best_model[1][0]*100:.2f}%)")

final_results = {
    'accuracies': {k: v[0] for k, v in results.items()},
    'training_times': {k: v[1] for k, v in results.items()},
    'best_model': best_model[0]
}

with open('model_results.pkl', 'wb') as f:
    pickle.dump(final_results, f)
print("Results saved")

Model Comparison Results:
Model                Accuracy     Time (s)  
--------------------------------------------------
Naive_Bayes          0.9359 (93.59%)  0.03    
Random_Forest        0.9394 (93.94%)  1.48    
LightGBM             0.9541 (95.41%)  6.06    
Complement_NB        0.8895 (88.95%)  0.01    
Passive_Aggressive   0.9447 (94.47%)  0.06    

Best Model: LightGBM (95.41%)
Results saved


In [18]:
# Cell 9: Data Summary
print("Data Summary:")
print("=" * 30)

non_zero_counts = X_train.getnnz(axis=1)
print(f"Average features per article: {non_zero_counts.mean():.1f}")
print(f"Data sparsity: {(1 - (X_train.nnz / (X_train.shape[0] * X_train.shape[1]))) * 100:.1f}%")

print(f"Training set - Real: {(y_train == 1).sum()}, Fake: {(y_train == 0).sum()}")
print(f"Test set - Real: {(y_test == 1).sum()}, Fake: {(y_test == 0).sum()}")

feature_names = vectorizer.get_feature_names_out()
print(f"Total features: {len(feature_names)}")

Data Summary:
Average features per article: 134.8
Data sparsity: 97.3%
Training set - Real: 5761, Fake: 1039
Test set - Real: 1441, Fake: 260
Total features: 5000
